# VISCERA — concept-supervised method (full pipeline, A6000/A100)
**VIS**ual-**C**oncept **E**ndoscopy **R**epresentation via **A**ttributes. Pipeline: SSL weight (dinov2.pth) → **Stage-1: concept-supervised pretrain on ~160k (35 clinical concepts)** with **L2-SP anchor** (keep SSL features) + GRL center-invariance → **Stage-2: downstream neo/ndbe** (PPV@90R tail loss + WiSE-FT + multi-seed ensemble).

**Drive `DRIVE_DIR` needs:** numbered data zips `0.zip..N.zip` (→ out/<dir>), `train.zip`/`val.zip` (→ out/train, out/val), `dinov2.pth`. No dataset.zip needed (labels come from the JSONs). Runtime → **GPU**.

In [ ]:
import torch; print(torch.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install timm==1.0.27 scikit-learn

In [ ]:
REPO_URL = 'https://github.com/HuynhDoTanThanh/RARE2026.git'   # <-- your repo
%cd /content
!rm -rf rare && git clone $REPO_URL rare
%cd /content/rare

In [ ]:
# ---- extract data from Drive (numbered chunks -> out/<dir>; train/val.zip -> out/train,out/val) ----
from google.colab import drive; drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/RARE_LG'   # <-- your Drive folder
import glob, os, zipfile, shutil
assert os.path.exists(f'{DRIVE_DIR}/dinov2.pth'), f'upload dinov2.pth to {DRIVE_DIR}'
shutil.copy(f'{DRIVE_DIR}/dinov2.pth', 'dinov2.pth'); print('dinov2.pth OK')
os.makedirs('out', exist_ok=True)
def extract_chunk(zpath):
    with zipfile.ZipFile(zpath) as zf:
        tops = {p.split('/')[0] for p in zf.namelist() if p.strip('/')}
        target = f"out/{os.path.splitext(os.path.basename(zpath))[0]}" if ('images' in tops or 'labels' in tops) else ('.' if tops=={'out'} else 'out')
        os.makedirs(target, exist_ok=True); zf.extractall(target)
for z in [p for p in sorted(glob.glob(f'{DRIVE_DIR}/*.zip')) if 'dataset' not in os.path.basename(p).lower()]:
    extract_chunk(z); print('extracted', os.path.basename(z))
print('out dirs:', len(glob.glob('out/*/labels')), '| train labels:', len(glob.glob('out/train/labels/*.json')), '| train imgs:', len(glob.glob('out/train/images/*')))

In [ ]:
# ---- labeled CSVs from the JSON labels (img = out/<split>/images/<name>.jpg) ----
import json, glob, os, csv
def build_csv(split):
    rows=[]
    for j in glob.glob(f'out/{split}/labels/*.json'):
        d=json.load(open(j))
        if int(d.get('label',-1))<0: continue
        name=d.get('name', os.path.splitext(os.path.basename(j))[0]); img=f'out/{split}/images/{name}.jpg'
        if os.path.exists(img): rows.append({'path':img,'center':d.get('center',''),'class':'','label':int(d['label'])})
    with open(f'{split}_colab.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=['path','center','class','label']); w.writeheader(); w.writerows(rows)
    print(f'{split}_colab.csv', len(rows),'pos=',sum(r['label'] for r in rows),'centers=',sorted({r['center'] for r in rows}))
build_csv('train')
try: build_csv('val')
except Exception as e: print('val skip', e)

In [ ]:
# ---- 35-concept target matrix (self-contained: unlabeled out/<dir> + labeled out/train from JSONs) ----
!python -m phase3.build_concept_targets --out phase3/cache/concept_targets.npz

## Stage-1 — concept-supervised pretraining (full-35 losses: certain / uncertain / smoothing)

LIGHT + L2-SP anchor → *shape*, don't overwrite SSL. Upgrades vs the old masked-BCE:
- **`--discrim full15`** — the full discriminative core (9 → 15; adds `color_heterogeneity, whitish_focal_area, vascular_irregularity, dilated_vessels, focal_abnormal_vessels, border_sharpness`). This is the real substance of "more labels".
- **certain** = class-balanced soft-BCE (`--pos_weight_cap 10` rebalances rare-positive concepts, e.g. `depression_ulceration` p=.055 → 10× — the term most likely to move PPV@90R).
- **uncertain** (`--unc_w 0.1`) = prior-pull on `not_assessable` cells instead of hard-masking (never invents a label).
- **smoothing** (`--smooth_eps 0.05`) = target shrinks toward the per-concept prior so the head can't get more confident than the noisy VLM warrants (anti center-memorization).

Every knob **defaults to the old masked-BCE** → clean superset, ablatable via the flags. Note: **7 of the 35 concepts are dead constants** (value ≡ 0: modality/distance/view/landmark/interpretable_fraction/dominant_color/lesion_size) and are auto-dropped; context/acquisition concepts go to a **detached AUX head** so "all 35" never re-injects center style into the trunk.

In [ ]:
# Stage-1 v2: full discriminative core + certain/uncertain/smoothing losses (all knobs default OFF == old masked-BCE).
#   drop the last two lines of flags to reproduce the previous baseline encoder exactly.
# NOTE: the LOCO gate below is low-power (~49/78 held-out positives) — read the per-method CIs, not the point estimate.
!python -m phase3.pretrain_concept --targets phase3/cache/concept_targets.npz \
    --unfreeze 3 --epochs 15 --bs 192 --lr 1e-4 --grl 1.0 --l2sp 1.0 --workers 8 \
    --discrim full15 --context_route detach \
    --certain_floor 0.7 --smooth_eps 0.05 --unc_w 0.1 --pos_weight_cap 10 \
    --out concept_encoder.pt

## Stage-2 — downstream from the concept encoder → 3-seed ship

Recipe = **fine-tune last-6 blocks + WiSE-FT 0.7** (the config behind exp1 = leaderboard PPV@90R **0.0181**). WiSE-FT interpolates the fine-tuned backbone back toward the SSL init, which **recovers the out-of-distribution robustness of linear-probing** (Kumar 2022: full FT distorts pretrained features under shift; Wortsman 2022 WiSE-FT is the fix) while keeping FT's capacity. So the old *head-only vs fine-tune* A/B was just the two ends of one knob — **`--head-only` ≈ `--wise-ft 0`** (frozen backbone, head only). If you want the LP↔FT tradeoff, sweep `--wise-ft {0.3,0.5,0.7,0.9}`, not a separate code path.

Ship trains on **both centers** (`--holdout none`), 3 seeds → ensemble. ⚠️ **New-center validation is deliberately NOT in this path** (the shipped model uses all data). Decide levers on an *external new-center proxy* (**RARE25-val**, the set the platform reports each submission) — the same-center val cell below is optimistic (it said 0.65 vs the real 0.018).

In [ ]:
# ---- LOCO GATE (local, free): validate BOTH decisions on the NEW-CENTER proxy BEFORE the ONE submission ----
# GATE ONLY; never shipped (ship cell uses --holdout none). Decision = PAIRED bootstrap gate on AUROC/AUPRC (ev.gate).
# 3 legs per holdout direction (6 finetune runs total):
#   cbase     = concept-init + baseline recipe (mild aug)
#   cbundle   = concept-init + bundle: +SWAD +OHEM + STRONG AUG   <- the intended ship
#   sslbundle = raw-SSL      + bundle                              <- does Stage-1 concept-init beat raw SSL?
# NOTE on the 2-center limit: SWAD and STRONG AUG are DOMAIN-RANDOMIZATION levers whose benefit is on UNSEEN
# centers, which a 2-center LOCO CANNOT measure. So for them LOCO is a DO-NO-HARM FLOOR (adopt unless it FAILs),
# not a proof-of-benefit. Hard-neg mining etc. (benefit visible at 2 centers) would instead need a PASS.
BASE   = '--unfreeze 6 --wise-ft 0.7 --epochs 30 --bs 96 --loss bce+rank+pauc --warmup 2'
BUNDLE = BASE + ' --swad --swad-last-n 5 --ohem-k 8 --aug strong'   # strong = endoscopy-curated RandAugment m6/mstd0.6
LEGS = {'cbase': ('--init concept_encoder.pt', BASE),
        'cbundle': ('--init concept_encoder.pt', BUNDLE),
        'sslbundle': ('', BUNDLE)}                          # '' = no --init = raw SSL teacher
for hold in ['center_2', 'center_1']:                       # both LOCO directions
    print(f'\n================ LOCO holdout={hold} ================')
    for name, (ini, flags) in LEGS.items():
        print(f'--- {name} ---')
        !python -m phase3.finetune --train-csv train_colab.csv {ini} --seed 0 \
            --holdout {hold} {flags} --out loco_{name}_{hold}.pt

# ---- PAIRED ACCEPTANCE GATES (Δ on SHARED resamples of the held-out center; legs share frame order) ----
import numpy as np, phase3.evaluate as ev
def leg(hold, name):
    d = np.load(f'loco_{name}_{hold}_loco.npz', allow_pickle=True); return d['y'], d['c'], d['s']
lever_fail, enc_fail = [], []
for hold in ['center_2', 'center_1']:
    y, c, s_cbundle = leg(hold, 'cbundle')
    _, _, s_cbase = leg(hold, 'cbase'); _, _, s_ssl = leg(hold, 'sslbundle')
    print(f"\n================ holdout {hold}  (held-out pos={int((y==1).sum())}) ================")
    print('  [LEVERS]  bundle(+SWAD+OHEM+strong-aug) vs baseline, concept encoder  (DO-NO-HARM floor):')
    for m in ('auroc', 'auprc'):
        g = ev.gate(y, s_cbundle, s_cbase, center=c, metric=m, B=2000)
        if g['verdict'] == 'FAIL': lever_fail.append((hold, m))
        print(f"    {m}: Δ={g['delta']:+.4f} CI[{g['lo']:+.4f},{g['hi']:+.4f}] -> {g['verdict']}")
    print('  [ENCODER] concept-init vs raw-SSL, bundle recipe:')
    for m in ('auroc', 'auprc'):
        g = ev.gate(y, s_cbundle, s_ssl, center=c, metric=m, B=2000)
        if g['verdict'] == 'FAIL': enc_fail.append((hold, m))
        print(f"    {m}: Δ={g['delta']:+.4f} CI[{g['lo']:+.4f},{g['hi']:+.4f}] -> {g['verdict']}")
print('\n================ SHIP DECISIONS ================')
print(f"  LEVERS  {'FAIL '+str(lever_fail)+' -> re-run cbundle with --aug mild to isolate; if still FAIL ship BASELINE' if lever_fail else 'no FAIL -> keep the bundle (SWAD+OHEM+strong-aug)'}")
print(f"  ENCODER {'FAIL '+str(enc_fail)+' -> ship RAW SSL (drop --init from the ship cell)' if enc_fail else 'no FAIL -> keep --init concept_encoder.pt'}")

In [ ]:
# ---- HARD-NEG MINING (final method) -> builds phase3/cache/unl_mined.txt for the ship cell ----
# One-sided PU: demote the confusable-NEGATIVE FP tail that decides PPV@90R. Two complementary sources, UNIONed:
#   (b) CTM  = concept-confounded hard negatives (offline, from the 35-concept matrix) — PU-guarded by the
#              decisive-hallmark ceiling (drops likely unlabeled POSITIVES) + --skip-top.
#   (c) model-in-the-loop = a round-0 scoring model's top false-positives among VLM-negative frames.
# NOTE (honest): a clean 2-center LOCO gate for mining is blocked by the missing dir->center map — mined negs
# would leak the held-out center's distribution. So mining relies on the PU guards + the leaderboard, NOT the gate.
# (a) manifest of the ~168k unlabeled VLM-scored pool (needs out/<dir> present)
!python -m phase3.mine_hardneg --confneg-sample 30000 --suspicious-top 5000
# (b) CTM concept-confounded negatives -> phase3/cache/unl_conceptFP.txt
!python -m phase3.mine_hardneg --concept-rank phase3/cache/concept_targets.npz --topn 4000 --skip-top 200
# (c) model-in-the-loop: round-0 scoring model (full bundle, both centers) -> phase3/cache/unl_modelFP.txt
!python -m phase3.finetune --train-csv train_colab.csv --holdout none --init concept_encoder.pt --seed 0 \
    --unfreeze 6 --wise-ft 0.7 --epochs 30 --swad --swad-last-n 5 --ohem-k 8 --aug strong --out ship_r0.pt
!python -m phase3.mine_hardneg --score-with ship_r0.pt --topn 4000 --skip-top 200
# (d) UNION (deduped) -> the final mined-negative list
import os
rd = lambda p: [l.strip() for l in open(p) if l.strip()] if os.path.exists(p) else []
ctm, mfp = rd('phase3/cache/unl_conceptFP.txt'), rd('phase3/cache/unl_modelFP.txt')
mined = sorted(set(ctm) | set(mfp))
open('phase3/cache/unl_mined.txt', 'w').write('\n'.join(mined) + '\n')
print(f'MINED: CTM={len(ctm)} + model-FP={len(mfp)} -> {len(mined)} unique negatives -> phase3/cache/unl_mined.txt')

In [ ]:
# ship: FULL FINAL METHOD on BOTH centers, 3 seeds -> ensemble.
#   concept-init + SWAD + OHEM tail-margin + STRONG AUG + MINED hard-negatives (CTM ∪ model-FP, one-sided PU).
# All train-time only -> shipped graph unchanged (viscera_model.py / container stay valid).
# Set flags to whatever the LOCO GATE did not FAIL; if the ENCODER gate FAILed, drop '--init' (ship raw SSL).
# CAVEAT (training review): --neg-cap couples to positive oversampling (more negs -> more batches/epoch -> each of
# the 127 positives is repeated more -> memorization). 4000 is moderate; if val/leaderboard shows overfitting,
# lower --neg-cap (e.g. 2000) or --epochs. (unl_mined.txt is produced by the HARD-NEG MINING cell above.)
STAGE2_FLAGS = ('--unfreeze 6 --wise-ft 0.7 --epochs 30 --swad --swad-last-n 5 --ohem-k 8 --aug strong '
                '--neg-list phase3/cache/unl_mined.txt --neg-cap 4000')
for s in [0, 1, 2]:
    !python -m phase3.finetune --train-csv train_colab.csv --holdout none --init concept_encoder.pt --seed {s} \
        {STAGE2_FLAGS} --bs 96 --loss bce+rank+pauc --warmup 2 --out ship_seed{s}.pt
import shutil
shutil.copy('concept_encoder.pt', f'{DRIVE_DIR}/concept_encoder.pt')
for s in [0, 1, 2]: shutil.copy(f'ship_seed{s}.pt', f'{DRIVE_DIR}/ship_seed{s}.pt')
print('saved encoder + 3 ensemble members to Drive')

## Inference (offline, per-image, ensemble + hflip-TTA)
```
!python -m phase3.infer --model ship_seed0.pt,ship_seed1.pt,ship_seed2.pt --images-dir <TEST_DIR> --out preds.csv
```
Knobs to tune: Stage-1 `--l2sp {0.3,1,3}` `--unfreeze {2,3}` `--epochs {15,25}`; Stage-2 `--wise-ft {0.3,0.5,0.7,0.9}` `--unfreeze {4,6,8}`. **Decide changes on an external new-center proxy (RARE25-val), not the same-center val below** — and only trust a change when its paired-Δ **AUROC/AUPRC** CI is clear of 0 (PPV@90R is too noisy at ~1% prevalence to rank configs).

## Val scoring → competition metric (PPV@90R @1% prevalence, bootstrap median + CI)
Scores `val_colab.csv` with the 3-seed ensemble and reports the leaderboard metric the same way the grader does (curve-point PPV@90R, 1% prevalence, bootstrap), plus AUROC/AUPRC. ⚠️ **This is SAME-CENTER (both centers were in training) → optimistic** (it read 0.65 vs the real new-center 0.018). Use it only as a smoke test that inference works; the honest new-center number comes from **RARE25-val / the leaderboard**, not here. Read the **CI**, not the point.

In [ ]:
# ---- score val with the 3-seed ensemble, then report the 5 trusted metrics ----
import os, csv, shutil, numpy as np
# restore ensemble from Drive if the runtime was reset
for s in [0, 1, 2]:
    if not os.path.exists(f'ship_seed{s}.pt') and os.path.exists(f'{DRIVE_DIR}/ship_seed{s}.pt'):
        shutil.copy(f'{DRIVE_DIR}/ship_seed{s}.pt', f'ship_seed{s}.pt')
assert all(os.path.exists(f'ship_seed{s}.pt') for s in [0, 1, 2]), 'ship_seed*.pt missing — run the ship cell or copy from Drive'
assert os.path.exists('val_colab.csv'), 'val_colab.csv missing — run the CSV-builder cell (needs out/val)'

!python -m phase3.infer --model ship_seed0.pt,ship_seed1.pt,ship_seed2.pt --csv val_colab.csv --out preds.csv

from phase3 import evaluate as ev
# preds.csv = name,score,label ; pull center from val_colab.csv by frame name
center_by = {os.path.splitext(os.path.basename(r['path']))[0]: r.get('center', '')
             for r in csv.DictReader(open('val_colab.csv'))}
P = list(csv.DictReader(open('preds.csv')))
y = np.array([int(r['label']) for r in P]); s = np.array([float(r['score']) for r in P])
cen = np.array([center_by.get(r['name'], '') for r in P])
print(f'val frames={len(y)}  pos={int(y.sum())}  centers={sorted(set(cen))}\n')

hdr = f"{'split':10s} {'n':>4s} {'pos':>3s} | {'PPV@90R':>8s} {'CI_low':>7s} {'CI_high':>7s} | {'AUROC':>6s} {'AUPRC':>6s}"
print(hdr); print('-' * len(hdr))


def line(tag, yv, sv, cv):
    r = ev.report_full(yv, sv, cv if len(set(cv)) > 1 else None, target=0.9, prevalence=0.01, B=2000)
    print(f"{tag:10s} {r['n']:>4d} {r['pos']:>3d} | {r['ppv90']:>8.3f} {r['ci_lo']:>7.3f} {r['ci_hi']:>7.3f} | "
          f"{r['auroc']:>6.3f} {r['auprc']:>6.3f}")


line('POOLED', y, s, cen)
for cc in sorted(set(cen)):
    mk = cen == cc
    if mk.sum() and y[mk].sum() > 0:
        line(cc, y[mk], s[mk], cen[mk])
print("\nPPV@90R = curve-point @1% prevalence (leaderboard metric, HIGH variance at few pos — read the CI).")
print("AUROC/AUPRC = threshold-free ranking, STABLE at few pos, but NOT the 1%-operating-point score.")
print("SAME-CENTER val is optimistic (ship saw both centers); the honest new-center number is RARE25-val / the leaderboard.")